In [1]:
pip install transformers torch

Note: you may need to restart the kernel to use updated packages.


In [3]:
from transformers import pipeline, AutoTokenizer, AutoModelForSequenceClassification

# Loading the model
model_name = "Sigma/financial-sentiment-analysis"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSequenceClassification.from_pretrained(model_name)
hf_sentiment = pipeline("sentiment-analysis", model=model, tokenizer=tokenizer)

Device set to use cpu


In [1]:
import praw
import json
import time
import random
import os
from datetime import datetime, timezone
from transformers import pipeline
import requests
from requests.adapters import HTTPAdapter
from urllib3.util.retry import Retry
import prawcore  # for specific exceptions 

#Resilient requests session for PRAW
def build_retry_session():
    session = requests.Session()
    retry = Retry(
        total=8,# overall retries
        connect=6,# DNS/connection retries
        read=6,# read retries
        status=6,# HTTP status retries
        backoff_factor=2.0,# 2s, 4s, 8s, 16s
        status_forcelist=[429, 500, 502, 503, 504],
        allowed_methods=["GET", "POST", "PUT", "DELETE", "HEAD", "OPTIONS", "TRACE"]
    )
    adapter = HTTPAdapter(max_retries=retry)
    session.mount("https://", adapter)
    session.mount("http://", adapter)
    return session

# Hugging Face sentiment analysis pipeline
sentiment_pipeline = pipeline(
    "sentiment-analysis",
    model="Sigma/financial-sentiment-analysis",
    device=-1  # set to 0 if GPU available
)

#Sentiment cache + batched processing 
SENT_CACHE = {}

def get_hf_sentiments(texts, batch_size=32):
    to_run, idx_map = [], {}
    results = [None] * len(texts)

    for i, t in enumerate(texts):
        t = (t or "").strip()
        if not t:
            results[i] = {"label": "UNKNOWN", "score": 0}
            continue
        if t in SENT_CACHE:
            results[i] = SENT_CACHE[t]
        else:
            idx_map[len(to_run)] = i
            to_run.append(t)

    if to_run:
        try:
            outs = sentiment_pipeline(
                to_run, truncation=True, max_length=512, batch_size=batch_size
            )
            label_map = {"LABEL_0": "negative", "LABEL_1": "neutral", "LABEL_2": "positive"}
            for j, r in enumerate(outs):
                mapped = {
                    "label": label_map.get(r['label'], "UNKNOWN"),
                    "score": round(float(r['score']), 3)
                }
                orig_idx = idx_map[j]
                results[orig_idx] = mapped
                SENT_CACHE[to_run[j]] = mapped
        except Exception:
            for j, txt in enumerate(to_run):
                mapped = get_hf_sentiment(txt)
                orig_idx = idx_map[j]
                results[orig_idx] = mapped
                SENT_CACHE[txt] = mapped
    return results

# Single fallback
def get_hf_sentiment(text):
    try:
        result = sentiment_pipeline(text[:512])[0]
        label_map = {
            "LABEL_0": "negative",
            "LABEL_1": "neutral",
            "LABEL_2": "positive"
        }
        return {
            "label": label_map.get(result['label'], "UNKNOWN"),
            "score": round(result['score'], 3)
        }
    except Exception:
        return {"label": "UNKNOWN", "score": 0}


# Reddit Scraper
class RedditScraper:
    def __init__(self, client_id, client_secret, user_agent, subreddits, post_limit=500, chunk_size=80, include_comments=True):
        session = build_retry_session()
        self.reddit = praw.Reddit(
            client_id=client_id,
            client_secret=client_secret,
            user_agent=user_agent,
            requestor_kwargs={"session": session},
            timeout=60  # a little higher timeout helps during hiccups
        )

        self.subreddits = subreddits
        self.post_limit = post_limit
        self.chunk_size = chunk_size
        self.include_comments = include_comments
        self.output_dir = "reddit_scrape_chunks"
        os.makedirs(self.output_dir, exist_ok=True)
        self.all_data = []
        self.file_index = 1

    def scrape(self):
        for sub in self.subreddits:
            print(f"Scraping r/{sub}...", flush=True)
            start_time = time.time()

            try:
                for count, post in enumerate(self.reddit.subreddit(sub).top(limit=self.post_limit), 1):
                    post_data = self._extract_post_data(post, sub)
                    self.all_data.append(post_data)

                    #Progress update every 50 posts
                    if count % 50 == 0:
                        elapsed = time.time() - start_time
                        print(f"[{sub}] {count} posts scraped... (elapsed ~{int(elapsed)}s)", flush=True)
                    
                    # Save in chunks 
                    if count % self.chunk_size == 0:
                        self._save_chunk(sub)
                        self.all_data = []
                        self.file_index += 1

                    time.sleep(0.25)  # per-post pause

                #Force save after subreddit
                if self.all_data:
                    self._save_chunk(sub)
                    self.all_data = []
                    self.file_index += 1

                time.sleep(random.uniform(3, 7))  # Between subreddits

            except Exception as e:
                print(f"Error in r/{sub}: {e}", flush=True)
                if "429" in str(e):
                    print("Rate limited. Sleeping for 60s...", flush=True)
                    time.sleep(60)
                else:
                    time.sleep(5)

    #Retry wrapper for comment fetching (handles DNS/connection resets)
    def _get_comments_with_retry(self, post, max_retries=6, base_sleep=2):
        last_err = None
        for attempt in range(1, max_retries + 1):
            try:
                post.comments.replace_more(limit=0)
                return list(post.comments.list())
            except (prawcore.exceptions.RequestException,
                    prawcore.exceptions.ResponseException,
                    prawcore.exceptions.ServerError,
                    prawcore.exceptions.OAuthException,
                    Exception) as e:
                last_err = e
                wait = min(base_sleep * (2 ** (attempt - 1)), 60)
                print(f"Error fetching comments for post {post.id} (attempt {attempt}/{max_retries}): {e}\n"
                      f"Retrying in {wait}s...", flush=True)
                time.sleep(wait)
        print(f"Skipping comments for post {post.id} after {max_retries} failed attempts.", flush=True)
        return []

    def _extract_post_data(self, post, subreddit):
        post_data = {
            'subreddit': subreddit,
            'title': post.title,
            'id': post.id,
            'url': post.url,
            'score': post.score,
            'text': post.selftext,
            'created_utc': post.created_utc,
            'created_at': datetime.fromtimestamp(post.created_utc, timezone.utc).strftime('%Y-%m-%d %H:%M:%S'),
            'num_comments': post.num_comments,
            'comments': []
        }

        if self.include_comments:
            try:
                #Use the retrying fetcher
                raw_comments = self._get_comments_with_retry(post)

                comments, texts = [], []
                for comment in raw_comments:  # all comments
                    if not hasattr(comment, "author") or comment.author is None:
                        continue
                    auth = str(comment.author).lower()
                    if any(x in auth for x in ("bot", "auto", "moderator")):
                        continue

                    body = (comment.body or "").strip()
                    if len(body) < 20 or body.lower() in ("[deleted]", "[removed]"):
                        continue

                    comments.append({
                        'id': comment.id,
                        'author': str(comment.author),
                        'body': body,
                        'score': comment.score,
                        'created_utc': comment.created_utc,
                        'created_at': datetime.fromtimestamp(comment.created_utc, timezone.utc).strftime('%Y-%m-%d %H:%M:%S')
                    })
                    texts.append(body)

                if texts:
                    sent_list = get_hf_sentiments(texts, batch_size=32)
                    for c, s in zip(comments, sent_list):
                        c['sentiment'] = s

                post_data['comments'].extend(comments)

            except Exception as e:
                print(f"Error processing comments for post {post.id}: {e}", flush=True)
                time.sleep(2)

        return post_data

    def _save_chunk(self, subreddit):
        file_path = os.path.join(self.output_dir, f"{subreddit}_batch_{self.file_index}.json")
        with open(file_path, "w", encoding="utf-8") as f:
            json.dump(self.all_data, f, indent=2)
        print(f" Saved {len(self.all_data)} posts to {file_path}", flush=True)


# Run Scraper
scraper = RedditScraper(
    client_id='94-KDboZSbfIo3SK3FDSzg',
    client_secret='6ktVmT5--Uj7CeubVKIJJCII2tFsEQ',
    user_agent='DataScrapper1811',
    subreddits=[
    'Altcoin','binance',
    'Bitcoin','BitcoinBeginners',
    'btc','cardano','CryptoCurrency',
    'CryptoMarkets','CryptoMoonShots',
    'CryptoTechnology','DeFi',
    'dogecoin','ethereum',
    'ethtrader','Ripple',
    'SatoshiStreetsBets',
    'solana',
    'Tronix',
    'XRP',
    'BitcoinMarkets'
],
    post_limit=500,
    chunk_size=100,   # saves every 100 posts
    include_comments=True
)

scraper.scrape()


Device set to use cpu


Scraping r/Tronix...
[Tronix] 50 posts scraped... (elapsed ~1685s)
[Tronix] 100 posts scraped... (elapsed ~3824s)
 Saved 100 posts to reddit_scrape_chunks\Tronix_batch_1.json
[Tronix] 150 posts scraped... (elapsed ~5341s)
[Tronix] 200 posts scraped... (elapsed ~6490s)
 Saved 100 posts to reddit_scrape_chunks\Tronix_batch_2.json
[Tronix] 250 posts scraped... (elapsed ~7548s)
[Tronix] 300 posts scraped... (elapsed ~8737s)
 Saved 100 posts to reddit_scrape_chunks\Tronix_batch_3.json
[Tronix] 350 posts scraped... (elapsed ~9543s)
[Tronix] 400 posts scraped... (elapsed ~10762s)
 Saved 100 posts to reddit_scrape_chunks\Tronix_batch_4.json
[Tronix] 450 posts scraped... (elapsed ~11638s)
[Tronix] 500 posts scraped... (elapsed ~12548s)
 Saved 100 posts to reddit_scrape_chunks\Tronix_batch_5.json
Scraping r/dogecoin...
[dogecoin] 50 posts scraped... (elapsed ~9484s)
[dogecoin] 100 posts scraped... (elapsed ~18780s)
 Saved 100 posts to reddit_scrape_chunks\dogecoin_batch_6.json
[dogecoin] 150 pos

In [ ]:
 pip install openai

In [13]:
from pathlib import Path
import json
import pandas as pd

ROOT = Path("reddit_scrape_chunks")  
print("[ROOT]", ROOT.resolve())

files = [p for p in ROOT.rglob("*.json") if p.is_file()]
print(f"[FOUND JSON FILES] {len(files)}")

posts_data, comments_data = [], []

for fp in files:
    try:
        if fp.stat().st_size == 0:
            print("[SKIP EMPTY]", fp)
            continue

        posts = json.loads(fp.read_text(encoding="utf-8", errors="ignore"))
        if not isinstance(posts, list):
            print("[SKIP NON-LIST]", fp)
            continue

        for post in posts:
            #Saving post info
            posts_data.append({
                "post_id": post.get("id"),
                "subreddit": post.get("subreddit"),
                "title": post.get("title"),
                "text": post.get("text"),
                "score": post.get("score"),
                "created_utc": post.get("created_utc"),
                "num_comments": post.get("num_comments"),
                "url": post.get("url")
            })

            #Saving each comment with link to post
            for c in post.get("comments", []):
                comments_data.append({
                    "post_id": post.get("id"),
                    "subreddit": post.get("subreddit"),
                    "post_title": post.get("title"),
                    "comment_id": c.get("id"),
                    "author": c.get("author"),
                    "body": c.get("body"),
                    "score": c.get("score"),
                    "created_utc": c.get("created_utc"),
                    "sentiment": c.get("sentiment", {})
                })

        if len(posts_data) % 500 == 0:
            print(f"[LOADED] {len(posts_data)} posts so far...")

    except Exception as e:
        print(f"[SKIP ERROR] {fp} -> {e}")

#Building DataFrames
df_posts = pd.DataFrame(posts_data)
df_comments = pd.DataFrame(comments_data)

print(f"[DONE] {len(df_posts)} posts, {len(df_comments)} comments loaded")

# Saving outputs
df_posts.to_csv("combined_posts.csv", index=False)
df_comments.to_csv("combined_comments.csv", index=False)

print("Saved  combined_posts.csv & combined_comments.csv")


[ROOT] C:\Users\Guppa\Poland Internship\reddit_scrape_chunks
[FOUND JSON FILES] 109
[LOADED] 500 posts so far...
[LOADED] 1000 posts so far...
[LOADED] 1500 posts so far...
[LOADED] 2000 posts so far...
[LOADED] 2500 posts so far...
[LOADED] 3000 posts so far...
[LOADED] 3500 posts so far...
[LOADED] 4000 posts so far...
[LOADED] 4500 posts so far...
[LOADED] 5000 posts so far...
[LOADED] 5500 posts so far...
[LOADED] 6000 posts so far...
[LOADED] 6500 posts so far...
[LOADED] 7000 posts so far...
[LOADED] 7500 posts so far...
[LOADED] 8000 posts so far...
[LOADED] 8500 posts so far...
[LOADED] 9000 posts so far...
[LOADED] 9500 posts so far...
[LOADED] 10000 posts so far...
[LOADED] 10500 posts so far...
[DONE] 10900 posts, 1299627 comments loaded
Saved  combined_posts.csv & combined_comments.csv


In [3]:
df_posts = pd.read_csv("combined_posts.csv")
len(df_posts)
df_posts.columns.tolist()

['post_id',
 'subreddit',
 'title',
 'text',
 'score',
 'created_utc',
 'num_comments',
 'url']

In [5]:
df_comments = pd.read_csv("combined_comments.csv")
len(df_comments) 
df_comments.columns.tolist()

['post_id',
 'subreddit',
 'post_title',
 'comment_id',
 'author',
 'body',
 'score',
 'created_utc',
 'sentiment']

In [4]:
import re, html
import pandas as pd

BOT_PAT   = re.compile(r'(automoderator|bot\b|_bot\b)', re.I)
URL_PAT   = re.compile(r'https?://\S+')
EMOJI_PAT = re.compile("[\U00010000-\U0010FFFF]", flags=re.UNICODE)
TICKER_PAT = re.compile(r"\b(btc|bitcoin|eth|ethereum|sol|solana|xrp|ada|doge|dogecoin|bnb|matic|trx|ltc)\b", re.I)
KEYWORDS = ('crypto','coin','token','market','bull','bear','pump','dump','rally',
            'sell-off','regulation','ath','binance','exchange')

def looks_spammy(text: str) -> bool:
    if not isinstance(text, str): return True
    t = text.strip()
    url_frac = len(URL_PAT.findall(t)) / max(1, len(t.split()))
    return (len(t) < 10) or (url_frac > 0.4) or (t.count('\n') > 20)

def normalize_text(s: str) -> str:
    if not isinstance(s, str): return ""
    s = html.unescape(s).lower()
    s = re.sub(URL_PAT, ' ', s)
    s = re.sub(r'@[A-Za-z0-9_]+', ' ', s)
    s = re.sub(r'#[A-Za-z0-9_]+', ' ', s)
    s = EMOJI_PAT.sub(' ', s)
    s = re.sub(r'[^a-z0-9$%\.\-\s]', ' ', s)
    s = re.sub(r'\s+', ' ', s).strip()
    return s

def is_relevant(clean_text: str) -> bool:
    if len(clean_text) < 50: return False
    if TICKER_PAT.search(clean_text): return True
    return any(k in clean_text for k in KEYWORDS)

def to_iso_utc(unix_ts):
    dt = pd.to_datetime(unix_ts, unit='s', utc=True, errors='coerce')
    return dt.strftime('%Y-%m-%dT%H:%M:%SZ') if pd.notnull(dt) else None


#Posts
def preprocess_posts(df_posts):
    df = df_posts.copy()
    df['raw_text'] = (df.get('title','').astype(str) + ' ' + df.get('text','').astype(str)).str.strip()
    df = df[df['raw_text'].notna() & (df['raw_text'].str.len() > 0)]

    df['clean_text'] = df['raw_text'].apply(normalize_text)
    df = df[~df['raw_text'].apply(looks_spammy)]
    df = df[df['clean_text'].apply(is_relevant)]

    df['iso_utc'] = df['created_utc'].apply(to_iso_utc)

    df = df.reset_index(drop=True)
    print('Preprocessing done (posts). Posts kept:', len(df))
    df.to_csv("filtered_reddit_posts.csv", index=False)
    df.to_json("filtered_reddit_posts.json", orient="records", lines=True)
    return df


#Comments
def preprocess_comments(df_comments):
    df = df_comments.copy()
    df['raw_text'] = df.get('body','').astype(str).str.strip()
    df = df[df['raw_text'].notna() & (df['raw_text'].str.len() > 0)]

    df['clean_text'] = df['raw_text'].apply(normalize_text)
    df = df[~df['raw_text'].apply(looks_spammy)]
    df = df[df['clean_text'].apply(is_relevant)]

    df['iso_utc'] = df['created_utc'].apply(to_iso_utc)

    df = df.reset_index(drop=True)
    print('Preprocessing done (comments). Comments kept:', len(df))
    df.to_csv("filtered_reddit_comments.csv", index=False)
    df.to_json("filtered_reddit_comments.json", orient="records", lines=True)
    return df


#Running Both
df_posts = pd.read_csv("combined_posts.csv")
df_comments = pd.read_csv("combined_comments.csv")

filtered_posts = preprocess_posts(df_posts)
filtered_comments = preprocess_comments(df_comments)

Preprocessing done (posts). Posts kept: 4954
Preprocessing done (comments). Comments kept: 434079
